##### WARNING
The following notebook is intended to be read only. Please do not modify the contents of this notebook.


# Overview
In this notebook, we will leverage the `SilverIngestionService` in Healthcare data solutions library to flatten selected FHIR resources in the `silver` lakehouse and to insert the resulting data into specified folder in ADLS GEN2 storage,
so it can be utilized by Customer Insights later.

As well this notebook optionally register the resulting dataset in the `customerinsights` lakehouse as tables, so it can be used by other tools like SQL endpoints to execute reporting in Fabric.

## Prerequisites
You need to have the following resources provisioned in your Azure subscription:
 - Azure Data Lake Storage Gen2 account

## Setup
Go to current `customerinsights` Lakehouse and follow instruction to create a shortcut to your storage account. [Instruction](https://learn.microsoft.com/en-us/fabric/onelake/create-adls-shortcut). Please make sure to have the shortcut name same as the value provided for the variable 'adsl_shortcut_name' during the deployment, default value is 'main'.

## Parameters

You can specify the folder name to store the ci tables under the vraiable 'all_entities_folder'.

## Run

Preferably, run the pipeline with the name <solution>_msft_customer_insights. You can also run on-demand or schedule this notebook to run periodically.

## Classes utilized

To run notebook we utilize OOB Healthcare data solutions classes `SilverIngestionService` and `FlattenManager`. You can find information about parameters in `msft_dm4h_bronze_silver_flatten` notebook.

In [ ]:
%run msft_config_notebook

In [ ]:
%run msft_config_notebook {"enable_spark_setup" : true, "enable_packages_mount" : false}

`inline_params` is a json dictionary of parameters(configuration values) which will take precedence and be use in place of the configuration values in the administration lakehouse

In [ ]:
inline_params = "{}"

In [ ]:
from microsoft.fabric.hls.hds.utils.utils import FolderPath
from microsoft.fabric.hls.hds.global_constants.global_constants import GlobalConstants as GC
from microsoft.fabric.hls.hds.utils.parameter_service import ParameterService
import json

# Shortcut name for the ADSL Gen2 container created as pre-requisite
inline_params_dict = json.loads(inline_params)
parameter_service = ParameterService(spark,
    workspace_name = workspace_name,
    admin_lakehouse_name = administration_database_name,
    one_lake_endpoint=one_lake_endpoint)

adsl_shortcut_name = parameter_service.get_activity_config_value("adsl_shortcut_name", "main")
# Folder to store all the customer insights tables to
all_entities_folder = 'all_entities'

customer_insights_database_name = "%%customer_insights_database_name%%"

#building the path to the config and schema files
transformation_config_root_dir = f'{data_manager_config_path}/{GC.INTERNAL_FOLDER}/{GC.FHIR_4}/{GC.TRANSFORMATION_CONFIGURATION_FOLDER}/ci'

ci_table_path = f"{FolderPath.get_fabric_files_path(workspace_name,one_lake_endpoint,customer_insights_database_name)}/{adsl_shortcut_name}/{all_entities_folder}"

In [ ]:
from microsoft.fabric.hls.hds.services.silver_ingestion_service import SilverIngestionService
from microsoft.fabric.hls.hds.flatten.flatten_manager import FlattenManager

inline_params_dict[GC.BRONZE_LAKEHOUSE_ID_KEY] =parameter_service.get_foundation_config_value(GC.SILVER_LAKEHOUSE_ID_KEY)
inline_params_dict[GC.SILVER_LAKEHOUSE_ID_KEY] = parameter_service.get_foundation_config_value(GC.CUSTOMER_INSIGHTS_LAKEHOUSE_ID_KEY)
inline_params_dict[GC.TARGET_TABLES_PATH_KEY] = ci_table_path
inline_params_dict[GC.CONFIG_PATH_KEY] = f'{transformation_config_root_dir}/{GC.FLATTEN_CONFIG_NAME}'

fm = FlattenManager(spark, 
    workspace_name = workspace_name, 
    solution_name = solution_name,
    one_lake_endpoint = one_lake_endpoint,
    transformation_config_root_dir = transformation_config_root_dir,
    unique_columns = ["Id"],
    source_modified_on_column = "LastUpdated")

silver_ingestion_service = SilverIngestionService(spark, 
    workspace_name = workspace_name,
    solution_name = solution_name,
    admin_lakehouse_name = administration_database_name,
    inline_params=inline_params_dict,
    one_lake_endpoint = one_lake_endpoint)

silver_ingestion_service.ingest_from_multiple_tables(silver_transformation_fn=fm.process_resource)

In [ ]:
supported_tables = ['Appointment', 'AppointmentParticipant', 'CarePlan', 'CarePlanCondition', 'Condition', 'Encounter', 'EncounterParticipant', 'Location', 'Patient', 'PatientAddress', 'Practitioner', 'PractitionerAddress', 'Goal']

for table in supported_tables:
    spark.sql(f"CREATE TABLE IF NOT EXISTS `{customer_insights_database_name}`.`{table}` USING DELTA LOCATION 'Files/{adsl_shortcut_name}/{all_entities_folder}/{table}'")

In [ ]:
mssparkutils.fs.unmount(packages_mount_name)